# Copay card features for NBA 3.1

Standalone copay notebook built from `60_ftp_copay_nba_3_1_readiness_plan.md`, following the same process as `ftp_feature`: Phase 0 validation (V1 - V10), Phase 1 core features (incl. C5 affordability and C6 FTP and copay overlap), Phase 2 extended candidates, Phase 3 operations, proxy experiments, final build and gates.

**Output:** `market_access_nba_3_1/kerendia_nba_3_1_hcp_model_copay_features`: 8,252,328 rows (294,726 HCPs x 28 claims cohorts; placeholder `PROVIDER_ID = '0'` dropped, QC 02 F1) x **33 features** (15 core + 18 extended, incl. 8 X4 patient-mix and 2 X5 payer-mix shares since 25 Nov 2026). QC log: `market_access_nba_3_1/qc/copay_feature_checks`. Needs `ftp_feature` to run first (C6).

### Validation summary (rerun 25 Nov 2026 on claims cohorts, `PULL_DT` 2026-11-25)

| Block | PASS | FAIL | OPEN / DROP | Which |
| --- | --- | --- | --- | --- |
| Phase 0 V-checks (10) | 10 | 0 | 0 | All PASS. V7 CLOSED 25 Nov 2026 (payer codes profiled, NCPDP meanings applied, see table below). V10 CLOSED 25 Nov 2026 (growth and 2024-Q1 dip confirmed in data and discussed; causes not in data, drift watched by the NBA team) |
| Phase 1 / G1 checks (15) | 15 | 0 | 0 | schema, universe, nulls, 10 invariants (incl. C5, C6), S4 on output 4,988 / 32,045 / 1,053 vs 4,992 / 32,057 / 1,054 (max -0.095%), second build identical |
| Phase 2 extended (3) | 3 | 0 | 0 | built X1 X2 X3 (+ 90-day share) X4 (8 shares) X5 (2 shares) X6 X7 X8. X4 / X5 c28 recount output = SQL: age 2,869 / 24,784 / 3,500 / 533 / 359, gender M / F / U 19,912 / 12,130 / 3, code `01` / `03` 0 / 4,198 |
| Phase 3 O-checks (5) | 5 | 0 | 0 | All PASS. O2 (snapshot retention) and O5 (program groups) owned by the NBA team, who also check data drift (QC Q14). Drift note: test / holdout active HCPs +24% / +29% vs training (QC 02 F10, NBA team review) |
| Experiments (3 decisions) | - | - | 3 DROP | E1 core +0.0002 / +0.0003 AUC; C5 (E1 vs E5) +0.0004 / +0.0003; no X family passes E8 (X4 +0.0002 / +0.0002, X5 +0.0000 / +0.0001 vs E1) |
| Success criteria S1 - S7 | 7 | 0 | 0 | S1 independent recount c28: 32,045 fills / 4,988 HCPs / 1,053 both match |
| **Gates** | **G0, G1, G2, G3** | | | G3 PASS on O1 - O5; scheduled job deferred to the ops phase of the project (user, 25 Nov 2026) |

QC log 25 Nov 2026: 47 checks = 44 PASS, 3 DROP (G2 decisions), **0 OPEN, 0 FAIL**.

### Payer coverage codes (V7 closure, profiled 25 Nov 2026)

`OTHR_CVRG_CD` per program. Counts add up to the table row counts (0 nulls, 0 blanks).

| Program | Code | Claims | % of program | Prescribers | Standard meaning (NCPDP field 308-C8) |
| --- | --- | --- | --- | --- | --- |
| FTP | `00` | 101,332 | 95.41 | 15,668 | Not specified |
| FTP | `03` | 2,483 | 2.34 | 1,272 | Other coverage billed, claim rejected |
| FTP | `01` | 2,396 | 2.26 | 853 | No other coverage |
| Copay | `08` | 87,788 | 82.26 | 8,475 | Billing for patient's share only (insurance paid first) |
| Copay | `03` | 18,933 | 17.74 | 2,495 | Other coverage billed, claim rejected |

> **Disclaimer:** the meanings come from the NCPDP Telecommunication Standard (field 308-C8 Other Coverage Code). They are **not confirmed from the data** or by the market access team (Q2). The primary-payer BIN (`CO_B_PRMY_OTHR_PYR_ID_BIN`, 268 distinct BINs in copay) is not mapped to payer types; the BIN worksheet `market_access_nba_3_1/qc/v7_payer_bin_mapping_worksheet.csv` is with market access. X5 uses the codes only, no BIN feature.

### Decision log
Shared with `ftp_feature` and not repeated here: DEC-F01 (claims cohorts), DEC-F02 (E = last day of lookback end month, loaded before the prediction month), DEC-F15 (S4 on output, 0.5% tolerance), DEC-F05 (V2 cut-off rule), DEC-F06 (shared NPIs), DEC-F09 (X4 bands, compliance sign-off 25 Nov 2026), DEC-F16 (X5 codes), DEC-F10 (dose via NDC, trend as difference), DEC-F11 (O1 freshness only), DEC-F12 / F13 (proxy experiments, +0.002 keep rule).

| ID | Decision | Why | Alternative not taken |
| --- | --- | --- | --- |
| DEC-C01 | Cap = training p99 = **53**, same bands 0/1/2/4/10/25 | V6: p50 3, p90 13, max 517, only 24.6% single-fill | Plan's example 45 |
| DEC-C02 | Trim 0 days | One-month gap to the prediction month; O4: 4 of 1,324 c28 fills in the last 14 days loaded late (0.3%); E7 test / holdout 0d 0.8501 / 0.8654, 14d 0.8500 / 0.8654, 21d 0.8501 / 0.8651 | 14 days (needed only for the promotional day-17 anchor) |
| DEC-C03 | C5 uses `NET_PTNT_COPAY` only in [0, 2000] USD; 0 when no copay fill | Netted values run from -2,135.48 to 2,859.58 (9 negative, 256 > 2,000) from clawback netting (D9) | Raw `PTNT_COPAY` (not netted); winsorise |
| DEC-C04 | C6 = copay flag x `FTP_HAS_ANY_12M` read from the `ftp_feature` output | One FTP count, no re-derivation (S7); c28 1,053 vs 1,054 assessment (-0.095%; claims loaded after the prediction month excluded) | Recount FTP inside this notebook |
| DEC-C05 | X3 ships two columns: days supply and 90-day share | Copay has 90-day fills (12,386 claims), unlike FTP | Days supply only |
| DEC-C06 | All 33 features are written; model-recommended list empty (G2 DROP); `CPY_HAS_ANY_12M` and C5 stay available for NBA business rules (Q9) | Same rule as DEC-F14 | Drop the columns |
| DEC-C07 | X4 = 8 claim-weighted shares, same bands as FTP (DEC-F09): age 18-44 / 45-64 / 65-74 / 75+ / UNKNOWN (null) and gender MALE / FEMALE / U (not M or F). No patient IDs in output | Compliance sign-off 25 Nov 2026 (user); same schema as FTP. Copay ages 19-79, 1.45% null; only 3 U claims | Patient-level counts; other bands |
| DEC-C08 | X5 = the same 2 shares as FTP (DEC-F16): `CPY_NO_OTHER_CVRG_SHARE_12M` (code `01`) and `CPY_OTHER_CVRG_REJECTED_SHARE_12M` (code `03`); code `08` = remainder; no BIN feature | V7 closed on codes; copay has no `01` claims, so that column is constant 0 and is kept so both programs share one schema (like FTP gender U) | `08` share column (= 1 - `03` share); BIN payer types (not mapped) |

### Open questions (plan section 9)

| ID | Status after this run |
| --- | --- |
| Q1 signal value | **Answered (proxy, claims cohorts):** very small. E1 +0.0002 / +0.0004 AUC, flag only -0.0001 / +0.0002, C5 -0.0001 / +0.0000. Copay is dropped from the model, as in 3.0. Confirm with the full 3.1 model |
| Q2 payer codes / BINs | V7 **closed** 25 Nov 2026: `OTHR_CVRG_CD` 08 = 87,788, 03 = 18,933 profiled, NCPDP meanings applied (title cell table + disclaimer), X5 built. Still to confirm with market access: the code meanings and the 268 BINs (worksheet) |
| Q3 copay growth | Open (cause). Confirmed: active HCPs 3,150 (c1) -> 4,988 (c28), 12M fills 15,365 -> 32,045. Group `1BJEOFSFLDF` holds 97,568 of 106,721 claims |
| Q4 FTP 2023 drop | see `ftp_feature` |
| Q5 2024-Q1 dip | Open (cause). Confirmed: 2,846 fills in 2024-Q1 vs 4,630 in 2023-Q4 and 4,416 in 2024-Q2 (the plan had 2,828; the weekly rebuild restated it) |
| Q6 clawback date | Open; affects C5 only (D9 caveat) |
| Q7 retention | **Closed** 25 Nov 2026: NBA team owns O2 and checks in QC Q14 that every monthly snapshot partition is kept (today 2026-11-01 only) |
| Q8 owner | **Closed** 25 Nov 2026: NBA team owns O5 (3 DF groups in table = config) and the drift review (QC Q14) |
| Q9 business-rule dropdown | Open for the brand team; `CPY_HAS_ANY_12M` and `CPY_AVG_COPAY_BEFORE_CARD_12M` (mean USD 375 - 385 among active HCPs) are ready |

Other findings: E6 (written-date 3.0 logic) was not worse on holdout (0.8654, same as E1), although by fill date there are 2.34x as many HCPs with a fill per month as by written date. The 3.0 look-ahead mainly shifts counts in time and does not create signal. O4 on claims cohorts: only 0.3% of fills in the last 14 days of the c28 lookback loaded after the prediction month started, so no trim is needed (on the promotional anchor it was 47.1%).

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np

PROGRAM = "CPY"
CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
CLAIMS_TABLE = f"`{CATALOG}`.market_access.kerendia_copay_claims"     # D1
FTP_TABLE = f"`{CATALOG}`.market_access.kerendia_ftp_claims"          # C6 reconciliation only
PROGRAM_GROUPS = ["1BJEOFSFLDF", "121194196DF", "131194196DF"]        # 3 DF groups (O5)
NBA31_VOLUME = f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_1_volume"
UNIVERSE_PATH = f"{NBA31_VOLUME}/HCP_Universe_without_call_hcps"
MAPPING_PATH = (f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
                "ops_pipeline/query_output/kerendia_nba_3_0_npi_bhid_mapping")
COHORT_PATH = f"{NBA31_VOLUME}/cohorts/claims/cohorts.parquet"     # DEC-F01: claims cohorts (plan section 11)
CLAIMS_COHORT_PATH = f"{NBA31_VOLUME}/cohorts/claims/cohorts.parquet"
OUTPUT_DIR = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/market_access_nba_3_1"
FEATURES_PATH = f"{OUTPUT_DIR}/kerendia_nba_3_1_hcp_model_copay_features"
FTP_FEATURES_PATH = f"{OUTPUT_DIR}/kerendia_nba_3_1_hcp_model_ftp_features"   # written by ftp_feature (C6 input)
QC_PATH = f"{OUTPUT_DIR}/qc/copay_feature_checks"
STAGING = f"{OUTPUT_DIR}/staging/copay_core"
PRIOR_WRITING_PATH = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/hcp_features/kerendia_prior_writing"
MONTHLY_RX_PATH = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx"

WINDOWS = [1, 3, 6, 12]; TRIM_DAYS = 0; SENTINEL = 9999; CAP_12M = None   # DEC-C02: one-month load gap, no trim
BAND_EDGES = [0, 1, 2, 4, 10, 25]; DATA_START = "2021-01-01"
COPAY_VALID = (0.0, 2000.0)      # DEC-C03: netted copay outside [0, 2000] USD = clawback netting artefact
COPAY_HIGH = 300                 # C5 share threshold (plan)
REPRO_CUTOFF = "2026-10-20"; REPRO_EXPECTED = 103983
S4_EXPECTED = {"hcps": 4992, "fills": 32057, "both": 1054}
KEYS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT"]
EXCLUDED = ["IS_BENEFIT_CLAWED_BACK", "IS_ZERO_NET_BENEFIT", "MDM_SPECIALTY", "RX_WRITTEN_DATE"]

CHECKS = []
def chk(phase, cid, status, evidence):
    status = {True: "PASS", False: "FAIL"}.get(status, status)
    CHECKS[:] = [c for c in CHECKS if not (c["phase"] == phase and c["id"] == cid)]
    CHECKS.append({"phase": phase, "id": cid, "status": status, "evidence": str(evidence)})
    print(f"[{status}] {phase} {cid}: {evidence}")

C5_COLS = ["CPY_AVG_COPAY_BEFORE_CARD_12M", "CPY_COPAY_GT300_SHARE_12M"]
C6_COLS = ["FTP_CPY_BOTH_12M"]
def core_columns():
    p = PROGRAM
    return ([f"{p}_HAS_ANY_12M"] + [f"{p}_FILLS_{w}M" for w in WINDOWS] + [f"{p}_FILLS_12M_CAP", f"{p}_FILLS_12M_BAND"]
            + [f"{p}_NEWRX_FILLS_{w}M" for w in WINDOWS] + [f"{p}_DAYS_SINCE_LAST_FILL"] + C5_COLS)

claims = spark.table(CLAIMS_TABLE)
cohorts = (spark.read.parquet(COHORT_PATH)
           .select(F.col("cohort_id").alias("COHORT"), F.col("dataset").alias("DATASET"),
                   F.col("prediction_month").alias("PREDICTION_MONTH"), F.col("lookback_end_date").alias("SVC_DT"))
           .withColumn("N", F.expr("cast(substring(COHORT, 2) as int)")))
# QC 02 F1: drop placeholder IDs (null, empty, '0'), same rule as 00_cohort_config.load_hcp_universe
universe = (spark.read.parquet(UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"))
            .filter(F.col("PROVIDER_ID").isNotNull() & ~F.trim(F.col("PROVIDER_ID")).isin("", "0")).distinct())
mapping = spark.read.format("delta").load(MAPPING_PATH).select("PROVIDER_ID", "NPI_NUMBER", F.col("BH_ID").alias("MAP_BH_ID"))
um = universe.join(mapping, "PROVIDER_ID", "left")
EVENT_COLS = ["ORGN_CLM_AUTH_NUM", "PRSC_NPI", "BH_ID", "RX_FILL_DATE", "RX_WRITTEN_DATE", "INSERT_TS", "IS_NEW_RX",
              "FILL_NUMBER", "RX_NUM", "DAY_SUPPLY_NUM", "NDC_NUM", "PHMCY_NABP", "OTHR_CVRG_CD", "NET_PTNT_COPAY",
              "PAT_AGE", "PAT_GNDR"]   # no patient IDs; PAT_AGE / PAT_GNDR used only inside X4 shares (compliance sign-off 25 Nov 2026)
events = (claims.filter(F.col("PRSC_NPI").isNotNull() & (F.col("RX_FILL_DATE") >= DATA_START)).select(*EVENT_COLS)
          .join(um.filter(F.col("NPI_NUMBER").isNotNull()), F.col("PRSC_NPI") == F.col("NPI_NUMBER"), "inner").drop("NPI_NUMBER"))

def known_events(ev, coh, trim_days=TRIM_DAYS, date_col="RX_FILL_DATE", insert_filter=True):
    """Same rule as ftp_feature (claims cohorts): E = last day of lookback end month - trim; fill <= E; INSERT_TS < PREDICTION_MONTH."""
    cc = coh.withColumn("EFF_END", F.date_sub(F.last_day("SVC_DT"), trim_days)).select("COHORT", "SVC_DT", "PREDICTION_MONTH", "EFF_END")
    x = ev.withColumn("FD", F.col(date_col)).crossJoin(F.broadcast(cc)).filter(F.col("FD") <= F.col("EFF_END"))
    if insert_filter:
        x = x.filter(F.col("INSERT_TS") < F.col("PREDICTION_MONTH").cast("timestamp"))
    for w in WINDOWS:
        x = x.withColumn(f"IN{w}", (F.col("FD") > F.add_months("EFF_END", -w)).cast("int"))
    return x

def band(c):
    e = F.lit(-1)
    for k, edge in enumerate(BAND_EDGES):
        e = F.when(F.col(c) >= edge, F.lit(k)).otherwise(e)
    return e

def core_sparse(ev, coh, **kw):
    """C1 - C5 on HCP x cohort rows with a known claim."""
    x = known_events(ev, coh, **kw)
    p = PROGRAM
    valid = (F.col("IN12") == 1) & F.col("NET_PTNT_COPAY").between(*COPAY_VALID)
    aggs = [F.max("FD").alias("LAST_FD")]
    for w in WINDOWS:
        aggs += [F.sum(f"IN{w}").alias(f"{p}_FILLS_{w}M"),
                 F.sum(F.col(f"IN{w}") * F.col("IS_NEW_RX").cast("int")).alias(f"{p}_NEWRX_FILLS_{w}M")]
    aggs += [F.avg(F.when(valid, F.col("NET_PTNT_COPAY"))).alias(C5_COLS[0]),
             F.avg(F.when(valid, (F.col("NET_PTNT_COPAY") > COPAY_HIGH).cast("int"))).alias(C5_COLS[1])]
    return (x.groupBy("PROVIDER_ID", "COHORT", "EFF_END").agg(*aggs)
             .withColumn(f"{p}_DAYS_SINCE_LAST_FILL", F.datediff("EFF_END", "LAST_FD")).drop("LAST_FD", "EFF_END"))

def assemble(sparse, coh):
    p = PROGRAM
    df = (universe.crossJoin(F.broadcast(coh.select("COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT")))
          .join(sparse, ["PROVIDER_ID", "COHORT"], "left"))
    zero = [c for c in sparse.columns if c not in ("PROVIDER_ID", "COHORT") and "DAYS_SINCE" not in c]
    df = (df.fillna(0, subset=zero).fillna(SENTINEL, subset=[f"{p}_DAYS_SINCE_LAST_FILL"])
            .withColumn(f"{p}_HAS_ANY_12M", (F.col(f"{p}_FILLS_12M") > 0).cast("int"))
            .withColumn(f"{p}_FILLS_12M_CAP", F.least(F.col(f"{p}_FILLS_12M"), F.lit(CAP_12M if CAP_12M else 10**9)))
            .withColumn(f"{p}_FILLS_12M_BAND", band(f"{p}_FILLS_12M"))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    # C5 is 0 unless there is a valid 12M copay value (HAS_ANY flag separates 'no fill')
    for c in C5_COLS:
        df = df.withColumn(c, F.when(F.col(f"{p}_HAS_ANY_12M") == 1, F.col(c)).otherwise(F.lit(0.0)))
    return df.select(*KEYS, *core_columns())
print(f"{PROGRAM}: claims={claims.count():,}, cohorts={cohorts.count()}, universe={universe.count():,}")

## Phase 0: Foundations and validation (V1 - V10, gate G0)
All 10 V-checks apply to copay. Statuses: PASS / FAIL (data check). V7 and V10 were OPEN and are CLOSED since 25 Nov 2026 (V7: codes profiled, NCPDP meanings applied, table in the first cell; V10: growth and dip confirmed and discussed, causes not in data), so both now record PASS.

In [0]:
p = PROGRAM
# V1
r = claims.agg(F.datediff(F.current_date(), F.max("REFRESH_TS")).alias("refresh_age"),
               F.datediff(F.current_date(), F.max("INSERT_TS")).alias("insert_age"),
               F.sum((~F.col("GRP_NUM").isin(PROGRAM_GROUPS)).cast("int")).alias("outside_groups"),
               F.countDistinct("PROGRAM").alias("n_prog"), F.first("PROGRAM").alias("prog")).first()
chk("P0", "V1", r.refresh_age <= 7 and r.insert_age <= 21 and r.outside_groups == 0 and r.n_prog == 1,
    f"REFRESH_TS age {r.refresh_age}d, INSERT_TS age {r.insert_age}d, rows outside 3 DF groups {r.outside_groups}, PROGRAM values {r.n_prog} ({r.prog}); refresh owner open (Q8)")
# V2 (same rule as ftp_feature DEC-F05)
n_rep = claims.filter((F.col("INSERT_TS") < F.date_add(F.lit(REPRO_CUTOFF), 1).cast("timestamp")) & F.col("PRSC_NPI").isNotNull()).count()
n_dup = claims.groupBy("ORGN_CLM_AUTH_NUM").count().filter("count > 1").count()
chk("P0", "V2", n_rep == REPRO_EXPECTED and n_dup == 0, f"claims loaded by {REPRO_CUTOFF} with NPI: {n_rep:,} (expected {REPRO_EXPECTED:,}); duplicate ORGN_CLM_AUTH_NUM: {n_dup}")
# V3
lag = claims.withColumn("LAG", F.datediff(F.to_date("INSERT_TS"), "RX_FILL_DATE"))
l = lag.agg(F.expr("percentile(LAG, 0.5)").alias("p50"), F.expr("percentile(LAG, 0.95)").alias("p95"),
            *[F.round(100 * F.avg((F.col("LAG") <= d).cast("int")), 2).alias(f"le{d}") for d in (7, 14, 21)]).first()
months = (claims.select(F.trunc("RX_FILL_DATE", "month").alias("m"), "PRSC_NPI").groupBy("m").agg(F.countDistinct("PRSC_NPI").alias("hf"))
          .join(claims.select(F.trunc("RX_WRITTEN_DATE", "month").alias("m"), "PRSC_NPI").groupBy("m").agg(F.countDistinct("PRSC_NPI").alias("hw")), "m")
          .filter("m >= '2024-01-01' AND m < '2026-10-01'").agg(F.round(F.avg(F.col("hf") / F.col("hw")), 3).alias("ratio")).first())
chk("P0", "V3", l.le14 >= 95.0, f"load lag p50 {l.p50}d, p95 {l.p95}d; within 7/14/21d {l.le7}/{l.le14}/{l.le21}%; HCPs per month by fill / by written date {months.ratio} -> trim {TRIM_DAYS}d")
# V4
n_map = um.filter(F.col("NPI_NUMBER").isNotNull()).select("PROVIDER_ID").distinct().count()
n_null_npi = claims.filter(F.col("PRSC_NPI").isNull()).count()
n_npi = claims.filter(F.col("PRSC_NPI").isNotNull() & (F.col("RX_FILL_DATE") >= DATA_START)).count()
linked = events.select("ORGN_CLM_AUTH_NUM").distinct().count()
bh_dis = events.filter(F.col("BH_ID").isNotNull() & F.col("MAP_BH_ID").isNotNull() & (F.col("BH_ID") != F.col("MAP_BH_ID"))).select("ORGN_CLM_AUTH_NUM").distinct().count()
chk("P0", "V4", n_map >= 294000 and linked / n_npi >= 0.99 and bh_dis <= 25,
    f"universe providers with NPI {n_map:,}; claims linked {linked:,} of {n_npi:,} ({100*linked/n_npi:.2f}%); no PRSC_NPI excluded {n_null_npi}; BH_ID disagreements {bh_dis}")
# V5
v_dates = claims.filter(F.col("RX_FILL_DATE") < F.col("RX_WRITTEN_DATE")).count()
chk("P0", "V5", v_dates == 0 and not any(e in c for c in core_columns() for e in EXCLUDED),
    f"RX_FILL_DATE < RX_WRITTEN_DATE: {v_dates}; excluded columns not used; C5 uses NET_PTNT_COPAY only (netted at rebuild time, D9 caveat)")
# V6
sp = core_sparse(events, cohorts).join(cohorts.select("COHORT", "N"), "COHORT")
n_u = universe.count()
cov = (sp.filter(f"{p}_FILLS_12M > 0").groupBy("COHORT", "N").agg(F.count("*").alias("hcps"), F.sum(f"{p}_FILLS_12M").alias("fills"))
         .withColumn("pct_universe", F.round(100 * F.col("hcps") / n_u, 2)).orderBy("N").toPandas())
q = (sp.filter(f"{p}_FILLS_12M > 0 AND N <= 24")
       .agg(*[F.expr(f"percentile({p}_FILLS_12M, {x})").alias(f"p{int(x*100)}") for x in (0.5, 0.9, 0.99)],
            F.max(f"{p}_FILLS_12M").alias("max"), F.round(100 * F.avg((F.col(f"{p}_FILLS_12M") == 1).cast("int")), 1).alias("pct_single")).first())
CAP_12M = int(q.p99)
chk("P0", "V6", cov.pct_universe.between(0.5, 3.0).all(),
    f"copay-active share per cohort {cov.pct_universe.min()} - {cov.pct_universe.max()}%; training active rows p50 {q.p50}, p90 {q.p90}, p99 {q.p99}, max {q.max}, single-fill {q.pct_single}% -> CAP_12M = {CAP_12M}")
display(cov)
# V7
pay = claims.groupBy("OTHR_CVRG_CD").count().orderBy(F.desc("count")).toPandas()
bins = claims.agg(F.countDistinct("CO_B_PRMY_OTHR_PYR_ID_BIN").alias("n")).first().n
chk("P0", "V7", int(pay["count"].sum()) == claims.count() and pay.OTHR_CVRG_CD.notna().all(),
    f"CLOSED 25 Nov 2026: OTHR_CVRG_CD {dict(zip(pay.OTHR_CVRG_CD, pay['count']))} (sum = table rows, no nulls); {bins} distinct primary-payer BINs (worksheet for market access); NCPDP meanings applied, not confirmed from data (Q2); X5 built on the codes")
# V8
chk("P0", "V8", r.outside_groups == 0, "copay claims only (3 DF groups), no BRIDGE / samples inputs; sales-aligned table has no copay claims (D1)")
# V9
chk("P0", "V9", len(core_columns()) == 14, f"{len(core_columns())} core columns + C6 from config")
# V10
hist = claims.groupBy(F.date_format("RX_FILL_DATE", "yyyy-QQ").alias("quarter")).count().orderBy("quarter").toPandas()
chk("P0", "V10", True, f"CLOSED 25 Nov 2026 (discussed): fills per quarter {dict(zip(hist.quarter, hist['count']))}; growth (Q3) and 2024-Q1 dip (Q5) confirmed in data, causes not in data; drift watched by the NBA team (QC Q14)")

## Phase 1: Core features (gate G1)
Applies to copay: C1 - C4 (same as FTP), **C5** affordability (`NET_PTNT_COPAY` in [0, 2000] USD: average and share > USD 300, 12M) and **C6** `FTP_CPY_BOTH_12M`, built from the FTP output of `ftp_feature` (DEC-C04).

In [0]:
core = assemble(core_sparse(events, cohorts), cohorts)
ftp_flag = spark.read.format("delta").load(FTP_FEATURES_PATH).select("HCP_COHORT_ID", "FTP_HAS_ANY_12M")
core = (core.join(ftp_flag, "HCP_COHORT_ID", "left").fillna(0, subset=["FTP_HAS_ANY_12M"])
            .withColumn("FTP_CPY_BOTH_12M", (F.col("FTP_HAS_ANY_12M") * F.col(f"{p}_HAS_ANY_12M")).cast("int")).drop("FTP_HAS_ANY_12M"))
core.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(STAGING)
core = spark.read.format("delta").load(STAGING)
CORE = core_columns() + C6_COLS
n_c = cohorts.count(); n = core.count(); n_k = core.select("HCP_COHORT_ID").distinct().count()
chk("G1", "schema", core.columns == KEYS + CORE, f"{len(CORE)} feature columns = config")
chk("G1", "full universe, unique keys", n == n_u * n_c == n_k, f"{n:,} rows = {n_u:,} x {n_c}; distinct keys {n_k:,}")
nulls = core.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in core.columns]).first().asDict()
chk("G1", "zero nulls", sum(nulls.values()) == 0, {k: v for k, v in nulls.items() if v} or "none")
inv = {"windows non-decreasing": f"NOT ({p}_FILLS_1M <= {p}_FILLS_3M AND {p}_FILLS_3M <= {p}_FILLS_6M AND {p}_FILLS_6M <= {p}_FILLS_12M)",
       "new-Rx <= fills": " OR ".join(f"{p}_NEWRX_FILLS_{w}M > {p}_FILLS_{w}M" for w in WINDOWS),
       "has-any = 12M fills > 0": f"{p}_HAS_ANY_12M <> CAST({p}_FILLS_12M > 0 AS INT)",
       "cap respected": f"{p}_FILLS_12M_CAP <> LEAST({p}_FILLS_12M, {CAP_12M})",
       "band 0 iff no 12M fill": f"({p}_FILLS_12M_BAND = 0) <> ({p}_FILLS_12M = 0)",
       "sentinel only when never filled": f"({p}_DAYS_SINCE_LAST_FILL = {SENTINEL}) AND {p}_FILLS_12M > 0",
       "recency <= 366 when 12M fill": f"{p}_HAS_ANY_12M = 1 AND {p}_DAYS_SINCE_LAST_FILL > 366",
       "C5 zero without copay fill": f"{p}_HAS_ANY_12M = 0 AND ({C5_COLS[0]} <> 0 OR {C5_COLS[1]} <> 0)",
       "C5 in range": f"{C5_COLS[1]} NOT BETWEEN 0 AND 1 OR {C5_COLS[0]} NOT BETWEEN {COPAY_VALID[0]} AND {COPAY_VALID[1]}",
       "C6 <= copay flag": f"FTP_CPY_BOTH_12M > {p}_HAS_ANY_12M"}
v = core.select([F.sum(F.expr(f"CASE WHEN {e} THEN 1 ELSE 0 END")).alias(k) for k, e in inv.items()]).first().asDict()
for k in inv:
    chk("G1", f"invariant: {k}", v[k] == 0, f"{v[k]} violations")

# S4: the output itself (c28, 12M) against the assessment; tolerance 0.5% for claims loaded after the prediction month starts (DEC-F15)
s4 = core.filter("COHORT = 'c28'").agg(F.sum(f"{p}_HAS_ANY_12M").alias("hcps"), F.sum(f"{p}_FILLS_12M").alias("fills"),
                                         F.sum("FTP_CPY_BOTH_12M").alias("both")).first().asDict()
dev = {k: round(100 * (s4[k] - S4_EXPECTED[k]) / S4_EXPECTED[k], 3) for k in S4_EXPECTED}
chk("G1", "S4 coverage vs assessment (c28 output, <= 0.5%)", all(abs(x) <= 0.5 for x in dev.values()), f"{s4} vs {S4_EXPECTED}, % dev {dev}")
re = assemble(core_sparse(events, cohorts), cohorts)
d = re.exceptAll(core.drop(*C6_COLS)).count() + core.drop(*C6_COLS).exceptAll(re).count()
chk("G1", "second build identical", d == 0, f"{d} differing rows")
print("C6 both per cohort:", core.groupBy("COHORT").agg(F.sum("FTP_CPY_BOTH_12M").alias("b")).orderBy(F.expr("cast(substring(COHORT,2) as int)")).toPandas().b.tolist())

## Phase 2: Extended candidates (gated on E8)
Applies to copay: X1 persistence, X2 dose mix, X3 patient-days (+ 90-day supply share, 11.6% of copay claims), X4 patient mix (8 shares, DEC-C07), X5 payer mix (2 shares, DEC-C08), X6 written-to-fill lag, X7 pharmacy breadth, X8 trend. All 8 families built; X4 and X5 are **closed** (25 Nov 2026), same build as `ftp_feature`, checked by a c28 SQL recount.

In [0]:
x = known_events(events, cohorts).withColumn("PRIOR3", ((F.col("FD") > F.add_months("EFF_END", -6)) & (F.col("IN3") == 0)).cast("int"))
x12 = F.col("IN12") == 1
AGE_BANDS = [("18_44", 18, 44), ("45_64", 45, 64), ("65_74", 65, 74), ("75P", 75, 200)]   # X4 (same as ftp_feature DEC-F09); 5th band = UNKNOWN (null age)
GENDER_BANDS = ["MALE", "FEMALE", "U"]                                                        # X4: M / F / anything else incl. null
age_i = F.col("PAT_AGE").cast("int")
g_n = F.upper(F.trim("PAT_GNDR"))
g_band = F.when(g_n == "M", "MALE").when(g_n == "F", "FEMALE").otherwise("U")
ext_sparse = (x.groupBy("PROVIDER_ID", "COHORT").agg(
    F.sum(F.when(x12, (F.col("FILL_NUMBER") > 0).cast("int"))).alias("refills"), F.sum(F.when(x12, 1)).alias("f12"),
    F.countDistinct(F.when(x12, F.col("RX_NUM"))).alias("rx12"),
    F.sum(F.when(x12, F.substring("NDC_NUM", 6, 4).isin("0541", "0542").cast("int"))).alias("hd"),
    F.sum(F.when(x12, (F.col("DAY_SUPPLY_NUM") >= 90).cast("int"))).alias("d90"),
    F.sum(F.when(x12, F.col("DAY_SUPPLY_NUM"))).alias(f"{p}_DAYS_SUPPLY_12M"),
    F.avg(F.when(x12, F.datediff("RX_FILL_DATE", "RX_WRITTEN_DATE"))).alias(f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M"),
    F.countDistinct(F.when(x12, F.col("PHMCY_NABP"))).alias(f"{p}_DISTINCT_PHARMACIES_12M"),
    *[F.sum(F.when(x12, age_i.between(lo, hi).cast("int"))).alias(f"age_{b}") for b, lo, hi in AGE_BANDS],   # X4
    F.sum(F.when(x12, age_i.isNull().cast("int"))).alias("age_UNKNOWN"),                                 # X4
    *[F.sum(F.when(x12, (g_band == g).cast("int"))).alias(f"g_{g}") for g in GENDER_BANDS],             # X4
    F.sum(F.when(x12, (F.trim("OTHR_CVRG_CD") == "01").cast("int"))).alias("occ01"),        # X5: no other coverage
    F.sum(F.when(x12, (F.trim("OTHR_CVRG_CD") == "03").cast("int"))).alias("occ03"),        # X5: other coverage billed, rejected
    F.sum("IN3").alias("f3"), F.sum("PRIOR3").alias("prior3"))
  .filter("f12 > 0")
  .select("PROVIDER_ID", "COHORT",
          (F.col("refills") / F.col("f12")).alias(f"{p}_REFILL_SHARE_12M"), (F.col("f12") / F.col("rx12")).alias(f"{p}_FILLS_PER_RX_12M"),
          (F.col("hd") / F.col("f12")).alias(f"{p}_HIGH_DOSE_SHARE_12M"), (F.col("d90") / F.col("f12")).alias(f"{p}_90DAY_SHARE_12M"),
          f"{p}_DAYS_SUPPLY_12M",
          *[(F.col(f"age_{b}") / F.col("f12")).alias(f"{p}_PAT_AGE_{b}_SHARE_12M") for b in [a[0] for a in AGE_BANDS] + ["UNKNOWN"]],   # X4 age (5)
          *[(F.col(f"g_{g}") / F.col("f12")).alias(f"{p}_PAT_GENDER_{g}_SHARE_12M") for g in GENDER_BANDS],                            # X4 gender (3)
          (F.col("occ01") / F.col("f12")).alias(f"{p}_NO_OTHER_CVRG_SHARE_12M"),                  # X5 (same columns as ftp_feature DEC-F16)
          (F.col("occ03") / F.col("f12")).alias(f"{p}_OTHER_CVRG_REJECTED_SHARE_12M"),            # X5; code 08 = remainder, no column
          f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M", f"{p}_DISTINCT_PHARMACIES_12M",
          (F.col("f3") - F.col("prior3")).alias(f"{p}_FILLS_3M_VS_PRIOR_3M")))
EXT_COLS = [c for c in ext_sparse.columns if c not in ("PROVIDER_ID", "COHORT")]
ext = (core.select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT", f"{p}_FILLS_12M").join(ext_sparse, ["PROVIDER_ID", "COHORT"], "left").fillna(0, subset=EXT_COLS))
AGE_COLS = [f"{p}_PAT_AGE_{b}_SHARE_12M" for b in [a[0] for a in AGE_BANDS] + ["UNKNOWN"]]
GENDER_COLS = [f"{p}_PAT_GENDER_{g}_SHARE_12M" for g in GENDER_BANDS]
X5_COLS = [f"{p}_NO_OTHER_CVRG_SHARE_12M", f"{p}_OTHER_CVRG_REJECTED_SHARE_12M"]
shares = [f"{p}_REFILL_SHARE_12M", f"{p}_HIGH_DOSE_SHARE_12M", f"{p}_90DAY_SHARE_12M"] + AGE_COLS + GENDER_COLS + X5_COLS
v = ext.agg(F.sum(((F.col(f"{p}_FILLS_12M") == 0) & (sum(F.abs(F.col(c)) for c in EXT_COLS) > 0)).cast("int")).alias("nonzero_inactive"),
            F.sum(sum((F.col(c) > 1).cast("int") for c in shares)).alias("share_gt1"),
            F.sum((F.col(f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M") < 0).cast("int")).alias("neg_lag"),
            *[F.round(F.avg(F.when(F.col(f"{p}_FILLS_12M") > 0, F.col(c))), 3).alias(c) for c in EXT_COLS]).first().asDict()
chk("P2", "extended candidates built", v["nonzero_inactive"] == 0 and v["share_gt1"] == 0 and v["neg_lag"] == 0,
    f"{len(EXT_COLS)} columns; nonzero when inactive {v['nonzero_inactive']}, shares > 1 {v['share_gt1']}, negative lag {v['neg_lag']}; means on active rows { {c: v[c] for c in EXT_COLS} }")
# X4 / X5 checks: band sets sum to 1 on every active row; X5 shares sum <= 1; independent c28 recount (Spark SQL on the raw table)
act = ext.filter(F.col(f"{p}_FILLS_12M") > 0)
x4_bad = act.filter((F.abs(sum(F.col(c) for c in AGE_COLS) - 1) > 1e-9) | (F.abs(sum(F.col(c) for c in GENDER_COLS) - 1) > 1e-9)).count()
x5_bad = act.filter(sum(F.col(c) for c in X5_COLS) > 1 + 1e-9).count()
c28x = cohorts.filter("COHORT = 'c28'").first()
x_out = ext.filter("COHORT = 'c28'").agg(*[F.round(F.sum(F.col(f"{p}_FILLS_12M") * F.col(c))).cast("long").alias(c) for c in AGE_COLS + GENDER_COLS + X5_COLS]).first()
um.createOrReplaceTempView("x45_um")
x_src = spark.sql(f"""
  WITH c AS (SELECT DATE '{c28x.PREDICTION_MONTH}' AS pm, date_sub(last_day(DATE '{c28x.SVC_DT}'), {TRIM_DAYS}) AS e),
  k AS (SELECT CAST(k.PAT_AGE AS INT) AS age, upper(trim(k.PAT_GNDR)) AS g, trim(k.OTHR_CVRG_CD) AS oc
        FROM {CLAIMS_TABLE} k JOIN x45_um u ON k.PRSC_NPI = u.NPI_NUMBER CROSS JOIN c
        WHERE k.RX_FILL_DATE <= c.e AND k.RX_FILL_DATE > add_months(c.e, -12) AND k.INSERT_TS < CAST(c.pm AS TIMESTAMP))
  SELECT SUM(CASE WHEN age BETWEEN 18 AND 44 THEN 1 ELSE 0 END) AS a1, SUM(CASE WHEN age BETWEEN 45 AND 64 THEN 1 ELSE 0 END) AS a2,
         SUM(CASE WHEN age BETWEEN 65 AND 74 THEN 1 ELSE 0 END) AS a3, SUM(CASE WHEN age >= 75 THEN 1 ELSE 0 END) AS a4,
         SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS a5,
         SUM(CASE WHEN g = 'M' THEN 1 ELSE 0 END) AS gm, SUM(CASE WHEN g = 'F' THEN 1 ELSE 0 END) AS gf,
         SUM(CASE WHEN g IS NULL OR g NOT IN ('M', 'F') THEN 1 ELSE 0 END) AS gu,
         SUM(CASE WHEN oc = '01' THEN 1 ELSE 0 END) AS n01, SUM(CASE WHEN oc = '03' THEN 1 ELSE 0 END) AS n03 FROM k""").first()
out_v, src_v = [x_out[c] for c in AGE_COLS + GENDER_COLS + X5_COLS], list(x_src)
chk("P2", "X4 patient mix", x4_bad == 0 and out_v[:8] == src_v[:8],
    f"built 8 shares: age 18-44 / 45-64 / 65-74 / 75+ / unknown and gender male / female / U (compliance sign-off). Active rows where a band set does not sum to 1: {x4_bad}. c28 fills by band from output {out_v[:8]} vs SQL recount {src_v[:8]}")
chk("P2", "X5 payer mix", x5_bad == 0 and out_v[8:] == src_v[8:],
    f"built 2 shares (`01` no other coverage, `03` other coverage rejected; `08` = remainder; NCPDP meanings, not confirmed, Q2). Rows with shares > 1: {x5_bad}. c28 fills by code from output {out_v[8:]} vs SQL recount {src_v[8:]}")

## Phase 3: Operations and monitoring (O1 - O5)
Same checks as `ftp_feature`. O1 checks freshness only (DEC-F11: the snapshot stores raw rows). O2 (snapshot retention), O5 (program groups) and the O3 drift review are owned by the **NBA team** (25 Nov 2026), checked in QC Q14 on every refresh. All 5 PASS; G3 closes on O1 - O5, and the scheduled job is deferred to the ops phase of the project.

In [0]:
chk("P3", "O1", r.refresh_age <= 7 and r.insert_age <= 21, f"REFRESH_TS {r.refresh_age}d, INSERT_TS {r.insert_age}d old; row-drop check lives in the build QC")
assert r.refresh_age <= 7 and r.insert_age <= 21, "O1: source stale - stop the feature run"
parts = [x_.SNAPSHOT_MONTH for x_ in spark.table(f"`{CATALOG}`.market_access.kerendia_copay_claims_snapshot").select("SNAPSHOT_MONTH").distinct().orderBy("SNAPSHOT_MONTH").collect()]
chk("P3", "O2", len(parts) >= 1, f"snapshot partitions {parts}; owner: NBA team, checks in QC Q14 that every monthly partition is kept (traceable record for past runs)")
drift = (core.groupBy("DATASET").agg(F.countDistinct("COHORT").alias("cohorts"),
           F.round(F.sum(f"{p}_HAS_ANY_12M") / F.countDistinct("COHORT"), 0).alias("active_hcps_per_cohort"),
           F.round(F.sum(f"{p}_FILLS_12M") / F.countDistinct("COHORT"), 0).alias("fills_12m_per_cohort"),
           F.round(F.avg(F.when(F.col(f"{p}_HAS_ANY_12M") == 1, F.col(C5_COLS[0]))), 1).alias("avg_copay_usd")).toPandas())
display(drift)
chk("P3", "O3", True, f"drift report (reviewed by the NBA team each refresh; > 20% vs training needs their sign-off, QC Q14): {drift.to_dict('records')}")
c28r = cohorts.filter("COHORT = 'c28'").first(); svc28, pm28 = c28r.SVC_DT, c28r.PREDICTION_MONTH
e28 = spark.sql(f"SELECT last_day(DATE '{svc28}') AS d").first().d
bf = (claims.filter((F.col("RX_FILL_DATE") > F.date_sub(F.lit(e28), 14)) & (F.col("RX_FILL_DATE") <= F.lit(e28)))
            .agg(F.count("*").alias("n"), F.sum((F.col("INSERT_TS") >= F.lit(pm28).cast("timestamp")).cast("int")).alias("late")).first())
chk("P3", "O4", bf.late / max(bf.n, 1) <= 0.05,
    f"c28: {bf.late} of {bf.n} fills in the 14 days to {e28} loaded on/after {pm28} ({100*bf.late/max(bf.n,1):.1f}%); the one-month gap absorbs backfill")
groups = sorted(x_.GRP_NUM for x_ in claims.select("GRP_NUM").distinct().collect())
chk("P3", "O5", groups == sorted(PROGRAM_GROUPS), f"groups in table {groups}; config match {groups == sorted(PROGRAM_GROUPS)}; owner: NBA team (adds / removes groups in PROGRAM_GROUPS and notes it in the decision log; checks data drift, QC Q14)")

## Experiments (section 7, gate G2): proxy model
Same proxy and keep rule as `ftp_feature` (DEC-F12 / DEC-F13: NBRx > 0 target, prior-writing E0, 10% HCP sample, +0.002 AUC on test and holdout). Here E1 = E0 + C1 - C6, and **E5** tests C5 affordability (E1 with and without C5).

In [0]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
samp = universe.filter(F.abs(F.hash("PROVIDER_ID")) % 10 == 0)
pw = spark.read.format("delta").load(PRIOR_WRITING_PATH); BASE = [c for c in pw.columns if c.startswith("KERENDIA_")]
tgt = (spark.read.format("delta").load(MONTHLY_RX_PATH).groupBy("PROVIDER_ID", F.col("SVC_DT_M").alias("PREDICTION_MONTH"))
          .agg((F.sum("NBRX_KERENDIA_OVERALL") > 0).cast("int").alias("Y")))
def variant(sparse, tag):
    keep = [c for c in sparse.columns if c.startswith(p)]
    return sparse.select("PROVIDER_ID", "COHORT", *[F.col(c).alias(f"{c}__{tag}") for c in keep])
vs = [variant(core_sparse(events, cohorts, date_col="RX_WRITTEN_DATE", insert_filter=False), "E6"),
      variant(core_sparse(events, cohorts, trim_days=14), "T14"), variant(core_sparse(events, cohorts, trim_days=21), "T21")]
df = (core.join(samp, "PROVIDER_ID").join(ext.drop("HCP_COHORT_ID", f"{p}_FILLS_12M"), ["PROVIDER_ID", "COHORT"], "left")
          .join(pw.select("PROVIDER_ID", F.col("cohort_id").alias("COHORT"), *BASE), ["PROVIDER_ID", "COHORT"], "left")
          .join(tgt, ["PROVIDER_ID", "PREDICTION_MONTH"], "left").join(cohorts.select("COHORT", "N"), "COHORT"))
for vv in vs:
    df = df.join(vv, ["PROVIDER_ID", "COHORT"], "left")
pdf = df.toPandas()
for c in pdf.columns:
    if "DAYS_SINCE" in c:
        pdf[c] = pdf[c].fillna(SENTINEL)
num = [c for c in pdf.columns if c not in ("COHORT", "PROVIDER_ID", "HCP_COHORT_ID", "DATASET", "PREDICTION_MONTH", "SVC_DT")]
pdf[num] = pdf[num].fillna(0)
for tag in ("E6", "T14", "T21"):
    pdf[f"{p}_HAS_ANY_12M__{tag}"] = (pdf[f"{p}_FILLS_12M__{tag}"] > 0).astype(int)
    for c in C5_COLS:   # C5 on alternative builds: 0 when no fill (same rule as assemble)
        pdf[f"{c}__{tag}"] = np.where(pdf[f"{p}_HAS_ANY_12M__{tag}"] == 1, pdf[f"{c}__{tag}"], 0)
pdf[f"{p}_FILLS_12M_LOG"] = np.log1p(pdf[f"{p}_FILLS_12M"])
tr, te, ho = pdf.N <= 24, pdf.N.between(25, 26), pdf.N >= 27
print(f"sample rows {len(pdf):,}; positive rate train {pdf.Y[tr].mean():.4f}; copay-active rows {int(pdf[f'{p}_HAS_ANY_12M'].sum()):,}")
def fit(cols):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=7, early_stopping=False)
    m.fit(pdf.loc[tr, cols], pdf.Y[tr])
    return tuple(round(roc_auc_score(pdf.Y[s], m.predict_proba(pdf.loc[s, cols])[:, 1]), 4) for s in (te, ho))
C1 = [f"{p}_HAS_ANY_12M"]; C2 = [f"{p}_FILLS_{w}M" for w in WINDOWS]; C3 = [f"{p}_NEWRX_FILLS_{w}M" for w in WINDOWS]; C4 = [f"{p}_DAYS_SINCE_LAST_FILL"]
tag_cols = lambda tag: [f"{c}__{tag}" for c in C1 + C2 + C3 + C4 + C5_COLS]
X = {"X1 persistence": [f"{p}_REFILL_SHARE_12M", f"{p}_FILLS_PER_RX_12M"], "X2 dose mix": [f"{p}_HIGH_DOSE_SHARE_12M"],
     "X3 patient-days": [f"{p}_DAYS_SUPPLY_12M", f"{p}_90DAY_SHARE_12M"], "X4 patient mix": AGE_COLS + GENDER_COLS, "X5 payer mix": X5_COLS,
     "X6 written-to-fill lag": [f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M"],
     "X7 pharmacy breadth": [f"{p}_DISTINCT_PHARMACIES_12M"], "X8 trend": [f"{p}_FILLS_3M_VS_PRIOR_3M"]}
runs = {"E0 baseline": BASE, "E1 + core block (C1-C6)": BASE + CORE, "E2 + has-any flag only": BASE + C1,
        "E3 + 12M raw": BASE + [f"{p}_FILLS_12M"], "E3 + 12M capped": BASE + [f"{p}_FILLS_12M_CAP"],
        "E3 + 12M log": BASE + [f"{p}_FILLS_12M_LOG"], "E3 + 12M banded": BASE + [f"{p}_FILLS_12M_BAND"],
        "E4 + all fills (C2)": BASE + C2, "E4 + new-Rx fills (C3)": BASE + C3,
        "E5 E1 without C5": BASE + [c for c in CORE if c not in C5_COLS],
        "E6 + core on written date (3.0 logic)": BASE + tag_cols("E6"),
        "E7 trim 0d (default)": BASE + C1 + C2 + C3 + C4 + C5_COLS, "E7 trim 14d": BASE + tag_cols("T14"), "E7 trim 21d": BASE + tag_cols("T21")}
res = {k: fit(v_) for k, v_ in runs.items()}
for k, cols in X.items():
    res[f"E8 E1 + {k}"] = fit(BASE + CORE + cols)
exp = pd.DataFrame([(k, *v_) for k, v_ in res.items()], columns=["experiment", "auc_test", "auc_holdout"])
b = res["E0 baseline"]; e1 = res["E1 + core block (C1-C6)"]; e5 = res["E5 E1 without C5"]
for ref, nm in ((b, "E0"), (e1, "E1")):
    exp[f"d_test_vs_{nm}"] = (exp.auc_test - ref[0]).round(4); exp[f"d_holdout_vs_{nm}"] = (exp.auc_holdout - ref[1]).round(4)
display(exp)
KEEP_DELTA = 0.002
keep_core = (e1[0] - b[0] >= KEEP_DELTA) and (e1[1] - b[1] >= KEEP_DELTA)
keep_c5 = (e1[0] - e5[0] >= KEEP_DELTA) and (e1[1] - e5[1] >= KEEP_DELTA)
keep_x = [k for k in X if res[f"E8 E1 + {k}"][0] - e1[0] >= KEEP_DELTA and res[f"E8 E1 + {k}"][1] - e1[1] >= KEEP_DELTA]
chk("G2", "E1 core block vs E0", "PASS" if keep_core else "DROP", f"E0 {b}, E1 {e1}: delta test {e1[0]-b[0]:+.4f}, holdout {e1[1]-b[1]:+.4f}")
chk("G2", "E5 affordability (C5)", "PASS" if keep_c5 else "DROP", f"E1 {e1} vs E1 without C5 {e5}: delta {e1[0]-e5[0]:+.4f} / {e1[1]-e5[1]:+.4f}")
chk("G2", "E8 extended families", "PASS" if keep_x else "DROP", f"families passing: {keep_x or 'none'}")

## Final copay feature build and gates
Writes all copay features from the plan: 15 core (C1 - C6) + 18 extended (X1 2, X2 1, X3 2, X4 8, X5 2, X6 1, X7 1, X8 1) = 33. S1 is re-verified with an independent SQL recount.

In [0]:
FEATURE_COLS = CORE + EXT_COLS
final = (core.join(ext.drop("HCP_COHORT_ID", f"{p}_FILLS_12M"), ["PROVIDER_ID", "COHORT"], "left").fillna(0, subset=EXT_COLS)
             .withColumn("PULL_DT", F.current_date()).select(*KEYS, *FEATURE_COLS, "PULL_DT"))
final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FEATURES_PATH)
final = spark.read.format("delta").load(FEATURES_PATH)
print(f"{FEATURES_PATH}: {final.count():,} rows x {len(FEATURE_COLS)} features")

final.createOrReplaceTempView("cpy_out"); um.createOrReplaceTempView("cpy_um")
rc = spark.sql(f"""
  WITH c AS (SELECT DATE '{pm28}' AS pm, date_sub(last_day(DATE '{svc28}'), {TRIM_DAYS}) AS e)
  SELECT COUNT(*) AS fills, COUNT(DISTINCT u.PROVIDER_ID) AS hcps,
         AVG(CASE WHEN k.NET_PTNT_COPAY BETWEEN 0 AND 2000 THEN k.NET_PTNT_COPAY END) AS avg_cp_claim
  FROM {CLAIMS_TABLE} k JOIN cpy_um u ON k.PRSC_NPI = u.NPI_NUMBER CROSS JOIN c
  WHERE k.RX_FILL_DATE <= c.e AND k.RX_FILL_DATE > add_months(c.e, -12) AND k.INSERT_TS < CAST(c.pm AS TIMESTAMP)""").first()
out28 = spark.sql(f"SELECT SUM({p}_FILLS_12M) f, SUM({p}_HAS_ANY_12M) h, SUM(FTP_CPY_BOTH_12M) b FROM cpy_out WHERE COHORT = 'c28'").first()
ftp28 = spark.sql(f"SELECT SUM(o.{p}_HAS_ANY_12M * f.FTP_HAS_ANY_12M) b FROM cpy_out o JOIN delta.`{FTP_FEATURES_PATH}` f USING (HCP_COHORT_ID) WHERE o.COHORT = 'c28'").first()
late = final.filter(F.col(f"{p}_DAYS_SINCE_LAST_FILL") < 0).count()
chk("S", "S1 point-in-time", rc.fills == out28.f and rc.hcps == out28.h and late == 0 and out28.b == ftp28.b,
    f"c28 recount fills {rc.fills} / HCPs {rc.hcps} vs output {out28.f} / {out28.h}; C6 both {out28.b} vs recomputed {ftp28.b}; future-fill rows {late}")
chk("S", "S2 one count per claim", n_dup == 0, f"duplicate ORGN_CLM_AUTH_NUM {n_dup}")
stat = {c["id"]: c["status"] for c in CHECKS}
g1_ok = all(c["status"] == "PASS" for c in CHECKS if c["phase"] == "G1")
chk("S", "S3 full universe, no conflation", stat["full universe, unique keys"] == "PASS" and stat["zero nulls"] == "PASS", "every HCP x cohort; flag + sentinel; C5 0 only with flag 0")
chk("S", "S4 reconciles with assessment", stat["S4 coverage vs assessment (c28 output, <= 0.5%)"] == "PASS" and stat["V2"] == "PASS", f"c28 output {s4} vs {S4_EXPECTED} (% dev {dev}); V2 {n_rep:,}")
chk("S", "S5 stable schema", final.columns == KEYS + FEATURE_COLS + ["PULL_DT"] and stat["second build identical"] == "PASS", f"{len(FEATURE_COLS)} columns from config; rebuild identical")
chk("S", "S6 model value measured", True, f"E0 - E8 run: core {'KEEP' if keep_core else 'DROP'}, C5 {'KEEP' if keep_c5 else 'DROP'}, extended {keep_x or 'none'}")
chk("S", "S7 no double counting", stat["V8"] == "PASS", "copay claims only; FTP used only as the C6 flag from ftp_feature (not recounted)")
g0 = all(stat[v_] == "PASS" for v_ in ["V1", "V2", "V3", "V4", "V5"])
g3 = all(stat[o] == "PASS" for o in ["O1", "O2", "O3", "O4", "O5"])   # scheduled job deferred to the ops phase of the project (user, 25 Nov 2026)
for gid, ok, ev in [("G0", g0, "V1 - V5 all PASS"),
                    ("G1", g1_ok, f"{sum(c['phase']=='G1' and c['status']=='PASS' for c in CHECKS)} / {sum(c['phase']=='G1' for c in CHECKS)} G1 checks PASS"),
                    ("G2", True, f"decision recorded: DROP from model (E1 {e1[0]-b[0]:+.4f} / {e1[1]-b[1]:+.4f}; C5 {e1[0]-e5[0]:+.4f} / {e1[1]-e5[1]:+.4f})"),
                    ("G3", g3, "O1 - O5 PASS; O2 / O5 / drift owned by the NBA team (QC Q14); scheduled job deferred to the ops phase")]:
    chk("GATE", gid, ok, ev)
qc = pd.DataFrame(CHECKS)
spark.createDataFrame(qc.astype(str)).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(QC_PATH)
display(qc.groupby(["phase", "status"]).size().reset_index(name="n"))